# Phase 1 — Dataset Exploration & Preparation

**Dataset:** [tmnam20/ViMedAQA](https://huggingface.co/datasets/tmnam20/ViMedAQA) (ACL 2024)  
**Task:** Abstractive Question Answering — Vietnamese Medical Domain  
**Output:**
- `../data/raw/vimedaq_full.json` — full dataset backup
- `../data/processed/train.json`, `val.json`, `test.json` — stratified splits
- `../data/eda/length_distributions.png` — EDA visualizations
- `../data/eda/dataset_stats.csv` — summary statistics

> All paths are relative to `notebooks/` so this notebook runs anywhere without reconfiguration.

## Cell 1 — Install Dependencies

In [1]:
# Install required packages
# Skip if already installed in your environment
import subprocess, sys

packages = ["datasets==2.19.0", "pandas==2.2.2", "matplotlib==3.9.0", "seaborn==0.13.2", "scikit-learn"]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("All packages installed.")

All packages installed.


## Cell 2 — Imports & Path Setup

In [2]:
import json
import os

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.model_selection import train_test_split

# All paths relative to notebooks/ directory
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))  # notebooks/
DATA_RAW       = os.path.join("..", "data", "raw")
DATA_PROCESSED = os.path.join("..", "data", "processed")
DATA_EDA       = os.path.join("..", "data", "eda")

os.makedirs(DATA_RAW,       exist_ok=True)
os.makedirs(DATA_PROCESSED, exist_ok=True)
os.makedirs(DATA_EDA,       exist_ok=True)

print("Paths configured:")
print(f"  Raw      : {os.path.abspath(DATA_RAW)}")
print(f"  Processed: {os.path.abspath(DATA_PROCESSED)}")
print(f"  EDA      : {os.path.abspath(DATA_EDA)}")

d:\000MINHTHONG\Junior - Semester II\SL\FinalPrjSL\medical\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Paths configured:
  Raw      : d:\000MINHTHONG\Junior - Semester II\SL\FinalPrjSL\medical\implementation\data\raw
  Processed: d:\000MINHTHONG\Junior - Semester II\SL\FinalPrjSL\medical\implementation\data\processed
  EDA      : d:\000MINHTHONG\Junior - Semester II\SL\FinalPrjSL\medical\implementation\data\eda


## Cell 3 — Load Dataset from HuggingFace

In [ ]:
# Load ViMedAQA from HuggingFace Hub
print("Loading tmnam20/ViMedAQA from HuggingFace Hub...")
dataset = load_dataset("tmnam20/ViMedAQA")

print("\nDataset info:")
print(dataset)
print("\nColumn names per split:")
for split in dataset:
    print(f"  {split}: {dataset[split].column_names}")

## Cell 4 — Save Raw Data Backup

In [ ]:
# Merge all splits into one list, tag each sample with its original split
all_data = []
for split in dataset:
    for item in dataset[split]:
        record = dict(item)
        record["_original_split"] = split
        all_data.append(record)

raw_path = os.path.join(DATA_RAW, "vimedaq_full.json")
with open(raw_path, "w", encoding="utf-8") as f:
    json.dump(all_data, f, ensure_ascii=False, indent=2)

print(f"Raw data saved: {len(all_data)} total samples -> {os.path.abspath(raw_path)}")

## Cell 5 — EDA: Dataset Statistics

In [ ]:
# Build a DataFrame for exploratory analysis
df = pd.DataFrame(all_data)

print("Columns  :", df.columns.tolist())
print("Shape    :", df.shape)
print("\nSample (first 2 rows):")
print(df[["question", "context", "answer", "topic"]].head(2).to_string())

# Null check
print("\nNull counts per column:")
print(df.isnull().sum())

# Topic distribution
print("\nOriginal split distribution:")
print(df["_original_split"].value_counts())

print("\nTopic distribution:")
print(df["topic"].value_counts())

# Word-level length statistics
df["question_len"] = df["question"].apply(lambda x: len(str(x).split()))
df["context_len"]  = df["context"].apply(lambda x: len(str(x).split()))
df["answer_len"]   = df["answer"].apply(lambda x: len(str(x).split()))

print("\nText length statistics (word count):")
print(df[["question_len", "context_len", "answer_len"]].describe().round(2))

## Cell 6 — EDA: Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("ViMedAQA Dataset — Exploratory Data Analysis", fontsize=14, fontweight="bold")

# (0,0) Topic distribution — bar chart
topic_counts = df["topic"].value_counts()
topic_counts.plot(kind="bar", ax=axes[0, 0], color="steelblue", edgecolor="black")
axes[0, 0].set_title("Topic Distribution")
axes[0, 0].set_xlabel("Topic")
axes[0, 0].set_ylabel("Count")
axes[0, 0].tick_params(axis="x", rotation=30)
for bar, count in zip(axes[0, 0].patches, topic_counts.values):
    axes[0, 0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        str(count),
        ha="center",
        va="bottom",
        fontsize=9,
    )

# (0,1) Question length — histogram
axes[0, 1].hist(df["question_len"], bins=30, color="coral", edgecolor="black")
axes[0, 1].set_title("Question Length (words)")
axes[0, 1].set_xlabel("Word count")
axes[0, 1].set_ylabel("Frequency")
axes[0, 1].axvline(df["question_len"].mean(), color="red", linestyle="--",
                   label=f"Mean: {df['question_len'].mean():.1f}")
axes[0, 1].legend()

# (1,0) Context length — histogram
axes[1, 0].hist(df["context_len"], bins=30, color="seagreen", edgecolor="black")
axes[1, 0].set_title("Context Length (words)")
axes[1, 0].set_xlabel("Word count")
axes[1, 0].set_ylabel("Frequency")
axes[1, 0].axvline(df["context_len"].mean(), color="darkgreen", linestyle="--",
                   label=f"Mean: {df['context_len'].mean():.1f}")
axes[1, 0].legend()

# (1,1) Answer length — histogram
axes[1, 1].hist(df["answer_len"], bins=30, color="purple", edgecolor="black")
axes[1, 1].set_title("Answer Length (words)")
axes[1, 1].set_xlabel("Word count")
axes[1, 1].set_ylabel("Frequency")
axes[1, 1].axvline(df["answer_len"].mean(), color="darkviolet", linestyle="--",
                   label=f"Mean: {df['answer_len'].mean():.1f}")
axes[1, 1].legend()

plt.tight_layout()

eda_png = os.path.join(DATA_EDA, "length_distributions.png")
plt.savefig(eda_png, dpi=150, bbox_inches="tight")
plt.show()
print(f"EDA chart saved -> {os.path.abspath(eda_png)}")

## Cell 7 — Stratified Train / Val / Test Split

In [ ]:
# Drop rows with null values in core columns
CORE_COLS = ["question", "context", "answer", "topic"]
df_clean = df.dropna(subset=CORE_COLS).reset_index(drop=True)
print(f"Samples after null removal: {len(df_clean)} (dropped {len(df) - len(df_clean)})")

# Check original split structure
print("\nOriginal split tag distribution:")
print(df_clean["_original_split"].value_counts())

# Stratified split: 80% train / 10% val / 10% test
# seed=42 for reproducibility
train_df, temp_df = train_test_split(
    df_clean,
    test_size=0.2,
    random_state=42,
    stratify=df_clean["topic"],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["topic"],
)

print(f"\nSplit sizes:")
print(f"  Train : {len(train_df)} ({len(train_df)/len(df_clean)*100:.1f}%)")
print(f"  Val   : {len(val_df)} ({len(val_df)/len(df_clean)*100:.1f}%)")
print(f"  Test  : {len(test_df)} ({len(test_df)/len(df_clean)*100:.1f}%)")

print("\nTrain topic distribution:")
print(train_df["topic"].value_counts())
print("\nVal topic distribution:")
print(val_df["topic"].value_counts())
print("\nTest topic distribution:")
print(test_df["topic"].value_counts())

## Cell 8 — Save Splits to processed/

In [ ]:
# Keep only the columns used by downstream training notebooks
KEEP_COLS = ["question", "context", "answer", "topic"]

split_map = {
    "train": train_df,
    "val":   val_df,
    "test":  test_df,
}

for name, split_df in split_map.items():
    out_path = os.path.join(DATA_PROCESSED, f"{name}.json")
    split_df[KEEP_COLS].to_json(
        out_path,
        orient="records",
        force_ascii=False,
        indent=2,
    )
    print(f"Saved {name}.json ({len(split_df)} samples) -> {os.path.abspath(out_path)}")

print("\nAll splits saved successfully.")

## Cell 9 — Save Summary Statistics CSV

In [ ]:
stats = {
    "total_samples":    len(df_clean),
    "train_size":       len(train_df),
    "val_size":         len(val_df),
    "test_size":        len(test_df),
    "num_topics":       df_clean["topic"].nunique(),
    "topics":           ",".join(sorted(df_clean["topic"].unique().tolist())),
    "avg_question_len": round(df_clean["question_len"].mean(), 2),
    "avg_context_len":  round(df_clean["context_len"].mean(), 2),
    "avg_answer_len":   round(df_clean["answer_len"].mean(), 2),
    "max_context_len":  int(df_clean["context_len"].max()),
    "max_question_len": int(df_clean["question_len"].max()),
    "max_answer_len":   int(df_clean["answer_len"].max()),
}

stats_path = os.path.join(DATA_EDA, "dataset_stats.csv")
pd.DataFrame([stats]).to_csv(stats_path, index=False)

print("Summary statistics:")
for k, v in stats.items():
    print(f"  {k}: {v}")
print(f"\nStats saved -> {os.path.abspath(stats_path)}")

print("\n" + "="*60)
print("IMPORTANT for Phase 3 tokenizer config:")
print(f"  max_context_len  = {stats['max_context_len']} words")
print(f"  max_answer_len   = {stats['max_answer_len']} words")
print("  => Use MAX_INPUT_LEN=512 for ViT5/mT5, 1024 for BARTpho")
print("  => Use MAX_TARGET_LEN=128 for ViT5/mT5, 256 for BARTpho")
print("="*60)

## Cell 10 — Phase 1 Checklist Verification

In [ ]:
# Verify all expected output files exist and are non-empty
expected_files = [
    os.path.join(DATA_RAW,       "vimedaq_full.json"),
    os.path.join(DATA_PROCESSED, "train.json"),
    os.path.join(DATA_PROCESSED, "val.json"),
    os.path.join(DATA_PROCESSED, "test.json"),
    os.path.join(DATA_EDA,       "length_distributions.png"),
    os.path.join(DATA_EDA,       "dataset_stats.csv"),
]

print("Phase 1 Output Verification:")
all_ok = True
for fp in expected_files:
    abs_fp = os.path.abspath(fp)
    exists = os.path.isfile(abs_fp)
    size   = os.path.getsize(abs_fp) if exists else 0
    status = "OK" if (exists and size > 0) else "MISSING"
    if status != "OK":
        all_ok = False
    print(f"  [{status}] {os.path.basename(abs_fp)} ({size:,} bytes)")

if all_ok:
    print("\nAll Phase 1 outputs verified. Ready for Phase 2 (Gemini baseline).")
else:
    print("\nSome outputs are missing. Re-run the cells above.")